# EvoAI Lab — Training Notebook

## "Most AI training asks if the model is right or wrong. We train for something harder: *is it right about being right?*"

---

This notebook trains **Llama-3-8B** using the EvoAI Lab self-improving loop.

The training target is **calibration** — eliminating **Zone C** (high confidence + wrong answer).

### What this notebook does:
1. Installs all dependencies
2. Sets up the Groq API key
3. Clones the EvoAI Lab repo and initialises the environment
4. Runs **50 training steps** of the self-improving loop
5. Plots the **dual-axis reward curve**
6. Runs **GRPO fine-tuning** on the collected data
7. Displays the **before/after calibration map**

### Zone definitions:
| Zone | Confidence | Correct? | What it means |
|------|-----------|----------|---------------|
| 🔴 Zone C | ≥ 7/10 | No | **Dangerous** — confident and wrong |
| 🟡 Zone B | < 7/10 | No | Wrong but appropriately uncertain |
| 🟢 Green  | Any    | Yes | Calibrated — right and knows it |

In [ ]:
# Cell 2 — Install all dependencies
# This cell must be run first. Runtime restart may be required after installation.
!pip install openenv groq sentence-transformers faiss-cpu httpx trl transformers unsloth datasets pydantic numpy -q
!pip install fastapi uvicorn -q
print("✅ All dependencies installed.")

In [ ]:
# Cell 3 — Set Groq API key from Colab secrets
# In Colab: click the key icon (🔑) in the left sidebar → add secret named GROQ_API_KEY
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab secrets.")
except Exception:
    # Fallback: paste your key directly (remove before sharing)
    os.environ["GROQ_API_KEY"] = "gsk_YOUR_KEY_HERE"
    print("⚠️  Using fallback API key. Set GROQ_API_KEY in Colab secrets for production.")

assert os.environ.get("GROQ_API_KEY"), "ERROR: GROQ_API_KEY is not set!"
print(f"   Key prefix: {os.environ['GROQ_API_KEY'][:8]}...")

In [ ]:
# Cell 4 — Clone the EvoAI Lab repository and import modules
import os
import sys

# Clone repo (skip if already present)
if not os.path.exists("evoai-lab"):
    !git clone https://github.com/YOUR_USERNAME/evoai-lab.git
else:
    print("Repo already present — skipping clone.")

%cd evoai-lab

# Add to path so imports work
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Create required directories
os.makedirs("data", exist_ok=True)
os.makedirs("plots", exist_ok=True)

# Import the environment
from backend.env.evoai_env import EvoAIEnv
from backend.core.pipeline import EvoAIPipeline

print("✅ EvoAI Lab imported successfully.")

In [ ]:
# Cell 5 — Run the self-improving training loop for 50 steps
import asyncio

env = EvoAIEnv()
env.reset()

print("🚀 Starting EvoAI Lab training loop (50 steps)...")
print(f"   Initial Zone C nodes: {env.state()['zone_c_count']}")
print(f"   Initial Zone B nodes: {env.state()['zone_b_count']}")
print("")

results = []

for i in range(50):
    result = await env.step()
    results.append(result)

    obs = result.get("observation", {})
    reward = result.get("reward", 0.0)
    probe = obs.get("last_probe", {})
    zone = probe.get("zone", "?")
    zone_icon = {"zone_c": "🔴", "zone_b": "🟡", "green": "🟢"}.get(zone, "⬜")
    skipped = result.get("info", {}).get("skipped", False)

    if skipped:
        print(f"Step {i+1:2d}: ⏭  skipped (teachers agreed)")
    else:
        sign = "+" if reward >= 0 else ""
        print(
            f"Step {i+1:2d}: {zone_icon} zone={zone:<8}  "
            f"reward={sign}{reward:.3f}  "
            f"Zone C={obs.get('zone_c_count',0)}  "
            f"Green={obs.get('green_count',0)}"
        )

print("\n✅ Training loop complete.")
final_state = env.state()
print(f"   Final Zone C nodes: {final_state['zone_c_count']}")
print(f"   Final Zone B nodes: {final_state['zone_b_count']}")
print(f"   Final Green nodes:  {final_state['green_count']}")

In [ ]:
# Cell 6 — Plot the dual-axis reward curve
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Extract rewards (skip skipped steps)
rewards = [
    r.get("reward", 0.0)
    for r in results
    if not r.get("info", {}).get("skipped", False)
]

steps = list(range(len(rewards)))
colors = ["#1D9E75" if r >= 0 else "#E24B4A" for r in rewards]

# Compute rolling mean (window=5)
window = 5
rolling_mean = [
    np.mean(rewards[max(0, i-window):i+1])
    for i in range(len(rewards))
]

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

# Bar chart
bars = ax.bar(steps, rewards, color=colors, alpha=0.8, width=0.7, zorder=2)

# Rolling mean line
ax.plot(steps, rolling_mean, color="#e6edf3", linewidth=1.5, linestyle="--",
        alpha=0.7, label=f"{window}-step rolling mean", zorder=3)

# Zero line
ax.axhline(0, color="rgba(255,255,255,0.3)", linewidth=0.8, zorder=1)

ax.set_xlabel("Training step", color="#8b949e", fontsize=10)
ax.set_ylabel("Reward", color="#8b949e", fontsize=10)
ax.set_title("EvoAI Lab — Dual-Axis Reward Curve", color="#e6edf3", fontsize=13, pad=12)
ax.set_ylim(-1.1, 1.1)
ax.tick_params(colors="#8b949e")
ax.spines[:].set_color("rgba(255,255,255,0.08)")
ax.grid(axis="y", color="rgba(255,255,255,0.04)", linestyle="-", zorder=0)

legend_patches = [
    mpatches.Patch(color="#1D9E75", alpha=0.8, label="Positive (calibrating)"),
    mpatches.Patch(color="#E24B4A", alpha=0.8, label="Negative (Zone C penalty)"),
]
ax.legend(
    handles=legend_patches + [plt.Line2D([0], [0], color="#e6edf3", linewidth=1.5,
                                          linestyle="--", label=f"{window}-step mean")],
    facecolor="#21262d", edgecolor="rgba(255,255,255,0.1)",
    labelcolor="#8b949e", fontsize=9, loc="upper left"
)

plt.tight_layout()
plt.savefig("plots/reward_curve.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

mean_reward = sum(rewards) / len(rewards) if rewards else 0.0
positive_count = sum(1 for r in rewards if r >= 0)

print(f"\n{'='*45}")
print(f"  Training Summary")
print(f"{'='*45}")
print(f"  Steps run:      {len(rewards)}")
print(f"  Mean reward:    {mean_reward:+.3f}")
print(f"  Positive steps: {positive_count}/{len(rewards)} ({positive_count/max(len(rewards),1):.0%})")
print(f"  Zone C start:   {results[0]['observation'].get('zone_c_count', '?')}")
print(f"  Zone C end:     {results[-1]['observation'].get('zone_c_count', '?')}")
print(f"  Green end:      {results[-1]['observation'].get('green_count', '?')}")
print(f"{'='*45}")
print("  📈 Plot saved to plots/reward_curve.png")

In [ ]:
# Cell 7 — Run GRPO fine-tuning
# This requires the training data collected in Cell 5.
# Requires a GPU runtime (T4 or better recommended).
import subprocess
import sys

# Flush dataset to disk first
env.pipeline.dataset_builder.flush_to_disk()
print("💾 Dataset flushed to data/training_pairs.jsonl")

from pathlib import Path
pairs_count = sum(1 for _ in open("data/training_pairs.jsonl")) if Path("data/training_pairs.jsonl").exists() else 0
print(f"   Training pairs available: {pairs_count}")

if pairs_count == 0:
    print("⚠️  No training pairs found. Run Cell 5 first to collect data.")
else:
    print("\n🔥 Starting GRPO fine-tuning...")
    # Use unsloth for 4-bit quantised training on free Colab T4
    os.environ["EVOAI_BASE_MODEL"] = "unsloth/llama-3-8b-bnb-4bit"
    !python backend/training/train_grpo.py
    print("\n✅ GRPO fine-tuning complete.")

In [ ]:
# Cell 8 — Display the before/after calibration map as a colour-coded table
# 🔴 = Zone C (dangerous), 🟡 = Zone B (uncertain wrong), 🟢 = Calibrated
import pandas as pd

final_state = env.state()
nodes = final_state["calibration_map"]["nodes"]

if not nodes:
    print("No nodes in calibration map yet.")
else:
    df = pd.DataFrame(nodes)

    zone_emoji = {"zone_c": "🔴 Zone C", "zone_b": "🟡 Zone B", "green": "🟢 Green"}
    df["Zone"] = df["zone"].map(zone_emoji).fillna("⬜ Unknown")
    df["Confidence Avg"] = df["confidence_avg"].round(2)
    df["Visits"] = df["visit_count"]

    display_df = df[["Zone", "topic", "question_type", "difficulty_tier", "Confidence Avg", "Visits"]]
    display_df = display_df.sort_values(["zone", "confidence_avg"], ascending=[True, False])
    display_df.columns = ["Zone", "Topic", "Question Type", "Difficulty", "Avg Confidence", "Visits"]

    print(f"\n{'='*75}")
    print(f"  EvoAI Lab — Final Calibration Map  ({len(nodes)} nodes)")
    print(f"{'='*75}")
    print(display_df.to_string(index=False))
    print(f"{'='*75}")

    zone_c_count = sum(1 for n in nodes if n["zone"] == "zone_c")
    zone_b_count = sum(1 for n in nodes if n["zone"] == "zone_b")
    green_count  = sum(1 for n in nodes if n["zone"] == "green")

    print(f"\n  🔴 Zone C: {zone_c_count}  🟡 Zone B: {zone_b_count}  🟢 Green: {green_count}")

    if zone_c_count == 0:
        print("  ✅ All Zone C nodes eliminated! The model is fully calibrated.")
    elif zone_c_count <= 3:
        print(f"  ⚡ Nearly calibrated — only {zone_c_count} Zone C node(s) remaining.")
    else:
        print(f"  ⚠️  {zone_c_count} Zone C nodes remain. Continue training to calibrate.")

    env.close()
    print("\n  💾 Environment closed and dataset flushed.")